# Lecture 01 · Gymnasium, Stable-Baselines3 and RL Zoo
**RLII_26 · Advanced Reinforcement Learning**

- Create and step a LunarLander environment.
- Locate rollout collection and PPO updates in Stable-Baselines3.
- Configure, run, evaluate, and debug a Zoo experiment.


**Content**: 
- Introduction to Gymnasium, Stable-Baselines3, and RL Baselines3 Zoo through PPO on LunarLander, connecting RL concepts to code, training, evaluation, and visualization.

**Goal**: 
- Enable students to navigate the repository and independently configure, run, and analyze reproducible reinforcement-learning experiments

In [ ]:
from pathlib import Path
import os
import sys
from importlib.metadata import version

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "rl_zoo3" / "exp_manager.py").is_file())
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache" / "matplotlib"))

In [ ]:
import gymnasium as gym
import pandas as pd
import numpy as np
import yaml
from IPython.display import Video, display
from stable_baselines3.common.env_util import make_vec_env
import rl_zoo3

ENV_ID = "LunarLander-v3"
COURSE = ROOT / "course" / "lecture_01"
VIDEOS = COURSE / "videos" / "generated"
print("Python:", sys.executable)
display(
    pd.DataFrame(
        [(name, version(name)) for name in ["gymnasium", "stable-baselines3", "rl_zoo3"]],
        columns=["Package", "Version"],
    )
)

## 1 · LunarLander: from random actions to a trained policy

- The video starts with a random policy, followed by PPO checkpoints.
- The task: land between the flags using four discrete actions.
- Which state variables are needed to control the landing?


In [ ]:
training_video = VIDEOS / "training_progress" / "training.mp4"
if training_video.is_file():
    display(Video(filename=str(training_video), embed=True, width=600))
else:
    print("Introductory video missing: check that videos/generated/training_progress/training.mp4 is present in your checkout.")

## 2 · One repository, three software layers

| Layer | Responsibility | Concrete object / entry point |
|---|---|---|
| Gymnasium | Task and environment interface | `gym.make("LunarLander-v3")` |
| Stable-Baselines3 | Learning algorithm | `PPO` |
| RL Baselines3 Zoo | Experiment infrastructure | `train.py`, `ExperimentManager`, Hyperparamters files |


### Main components:
| Repository component | Role |
|---|---|
| `train.py` | Thin command-line entry point |
| `rl_zoo3/` | Importable Python package |
| `hyperparams/ppo.yml` | Environment-specific PPO settings |
| `pyproject.toml` | Packaging, dependencies, optional extras, tooling |
| `uv.lock` | Resolved dependency versions |
| `.python-version` | Course Python version: 3.12 |
| `tests/`, `docs/` | Tests and upstream documentation |
| `course/` | Additive teaching material |

- A **module** is an importable Python file; a **package** groups modules.
- Editable installation connects imports to this checkout.
- A virtual environment isolates package installations.
- Git tracks history locally; GitHub hosts the shared remote repository.
- Basic workflow: `clone → pull → status → add → commit → push`.
- Run `git status` before committing; avoid committing generated experiment artifacts.


In [ ]:
print("Imported Zoo package:", Path(rl_zoo3.__file__).resolve())

## 3 · Markov decision model: reminder

$$
S_0\sim\mu,\qquad A_t\sim\pi_\theta(\cdot\mid S_t).
$$

$$
S_{t+1}\sim P(\cdot\mid S_t,A_t),\qquad R_{t+1}=R(S_t,A_t,S_{t+1}).
$$

- $\mu$: initial-state distribution; $\pi_\theta$: policy; $P$: transition kernel.
- For LunarLander, reward is computed from the transition and action.
- The current state and action determine the next-state distribution.

~~~text
state S_t → policy → action A_t → environment → state S_{t+1}, reward R_{t+1}
~~~


### What is Gymnasium?

- A Python library defining the environment API, task registry, and wrappers.
- `gym.make("LunarLander-v3")` constructs the task and its default wrappers.
- A wrapper adds behavior while preserving `reset()` and `step()`.

| Component | Responsibility |
|---|---|
| Gymnasium | API, task registry, environment implementations and wrappers |
| LunarLander task | Actions, state vector, reward and termination rules |
| Box2D | 2D physics: forces, collisions and joint motion |
| MuJoCo | Physics for articulated systems, e.g. HalfCheetah |
| SB3 PPO | Select actions and update the policy/value networks |

~~~text
PPO → Gymnasium API → LunarLander → Box2D
PPO → Gymnasium API → HalfCheetah → MuJoCo
~~~


### Important environment attributes and methods

| Attribute / method | Meaning / return value |
|---|---|
| `observation_space` | Structure, dtype, and allowed values of observations |
| `action_space` | Valid actions; `.sample()` samples a random action |
| `reset(seed=...)` | Starts an episode; returns `(observation, info)` |
| `step(action)` | Advances the task; returns `(next_observation, reward, terminated, truncated, info)` |
| `render()` | Visualizes the task; select `render_mode="rgb_array"` for frames or `"human"` for a window at construction |
| `close()` | Releases simulator/rendering resources |
| `unwrapped` | Base environment underneath Gymnasium wrappers |
| `spec` | Registry information when created with `gym.make`: e.g. ID and time limit |

- Reset before the first step and after an episode ends.
- `info` contains diagnostics; it is not automatically part of policy input.
- The simulator returns samples; an explicit transition matrix is not required.
- The environment seed and the seed for `action_space.sample()` are separate.

### RL notation → Python

| Mathematics | Python | Interpretation |
|---|---|---|
| $S_t$ | `observation` | State returned by the environment |
| $A_t$ | `action` | Control input |
| $R_{t+1}$ | `reward` | Immediate reward from this transition |
| $S_{t+1}$ | `next_observation` | Observation after the transition |
| $P,R$ | `env.step(action)` | Sample transition and compute reward |
| $\mu$ | `env.reset(seed=...)` | Sample an initial condition |
| $\pi_\theta$ | `model.policy` | Parameterized action distribution, implemented by a neural network |


In [ ]:
env = gym.make(ENV_ID)
try:
    print("Wrapper chain:", env)
    print("Observation space:", env.observation_space)
    print("Action space:", env.action_space)
    observation, info = env.reset(seed=0)
    env.action_space.seed(0)
    action = env.action_space.sample()
    next_observation, reward, terminated, truncated, info = env.step(action)
    display(
        pd.DataFrame(
            {
                "observation": observation,
                "next_observation": next_observation,
            },
            index=["x", "y", "vx", "vy", "angle", "angular velocity", "left leg contact", "right leg contact"],
        )
    )
    print(f"Action={action}, reward={reward:.3f}")
    print(f"terminated={terminated}, truncated={truncated}, info={info}")
finally:
    env.close()

### LunarLander: state, actions and reward

**State**:
| State components | Meaning |
|---|---|
| $x,y$ | Normalized horizontal and vertical position |
| $v_x,v_y$ | Scaled horizontal and vertical velocity |
| $\alpha,\omega$ | Angle and scaled angular velocity |
| $c_L,c_R$ | Ground-contact indicators for the two legs |

**Actions**:
| Action | Control |
|---:|---|
| 0 | Do nothing |
| 1 | Fire left orientation engine |
| 2 | Fire main engine |
| 3 | Fire right orientation engine |

$$
\Phi(S_t)=-100\sqrt{x_t^2+y_t^2}
-100\sqrt{v_{x,t}^2+v_{y,t}^2}
-100|\alpha_t|+10c_{L,t}+10c_{R,t}.
$$

For a nonterminal transition:

$$
R_{t+1}=\Phi(S_{t+1})-\Phi(S_t)
-0.30\,u_t^{\mathrm{main}}-0.03\,u_t^{\mathrm{side}}.
$$

- Position, speed and angle terms favor approaching the pad slowly and upright.
- Gaining leg contact adds to the score; losing contact subtracts from it.
- Reward is the **change** in score, not the score itself.
- $u_t^{\mathrm{main}},u_t^{\mathrm{side}}\in\{0,1\}$ indicate engine use.
- A crash or horizontal out-of-bounds event replaces this reward with −100.
- A settled/asleep lander replaces it with +100; this check comes last in the code.
- Reaching the time limit adds no special penalty.
- Logged episode return is the undiscounted sum of step rewards.


### Open the environment implementation

[LunarLander source](../../.venv/lib/python3.12/site-packages/gymnasium/envs/box2d/lunar_lander.py):
find `step()`, then `shaping` and `reward`.

[Gymnasium environment API](../../.venv/lib/python3.12/site-packages/gymnasium/core.py):
find `Env.reset()` and `Env.step()`.


### Episode boundaries: termination ≠ truncation

State value is the expected **future** discounted reward under policy $\pi$:

$$V^\pi(s)=\mathbb E_\pi[G_t\mid S_t=s].$$

| State / boundary | Future reward | Value |
|---|---|---|
| True terminal state (`terminated=True`) | Task is over; no further rewards in this episode | $V^\pi(s_{\mathrm{terminal}})=0$ |
| Nonterminal state | Task can continue and produce further rewards | $V^\pi(s)$ is generally not zero |
| External cutoff (`truncated=True`) | Collection stops, but the task state need not be terminal | Do not automatically set its value to zero |

- A nonterminal value can be positive, negative, or coincidentally zero.
- The reward **for entering** a terminal state can be nonzero: e.g. the landing reward +100.
- Value at that terminal state is nevertheless zero because it concerns rewards **after arrival**.
- A time limit is an external cutoff.
- Both signals end the interaction loop: `episode_finished = terminated or truncated`.

**Key distinction:** stopping an episode in code does not always mean reaching a terminal state of the task.


In [ ]:
# Force a one-step time limit so truncation is easy to observe.
limited_env = gym.make(ENV_ID, max_episode_steps=1)
try:
    observation, info = limited_env.reset(seed=0)
    next_observation, reward, terminated, truncated, info = limited_env.step(0)
    print({"terminated": terminated, "truncated": truncated, "episode_finished": terminated or truncated})
    assert truncated
finally:
    limited_env.close()

### Wrappers in our PPO–LunarLander experiment

A Gymnasium wrapper exposes the same environment interface and delegates to an inner environment,
while adding checks, limits, transformations, or statistics.

With the current `hyperparams/ppo.yml` and default Zoo CLI settings:

~~~text
PPO
 ↕
VecEnv                     SB3 vectorized container: 16 environment instances
 ├── Monitor                    training episode statistics
 │    └── PatchedTimeLimit       Zoo time-limit handling
 │         └── OrderEnforcing    require reset before stepping
 │              └── PassiveEnvChecker
 │                   └── LunarLander
 │                        └── Box2D physics world
 ├── same chain for environment 1
 └── ... up to environment 15
~~~

| Layer | Added by | Role |
|---|---|---|
| `PassiveEnvChecker` | Gymnasium `make` | Checks spaces, observations, and API return values at designated initial calls |
| `OrderEnforcing` | Gymnasium `make` | Enforces reset-before-use ordering |
| `PatchedTimeLimit` (subclass of `TimeLimit`) | Zoo patches Gymnasium `make` | Enforces the 1,000-step limit; preserves timeout metadata without adding a timeout flag to an otherwise terminated episode |
| `Monitor` | SB3 `make_vec_env`, called by Zoo | Records episode return, length, elapsed time; writes training `*.monitor.csv` files when given a log directory |
| `DummyVecEnv` | SB3 `make_vec_env`, selected by Zoo | Steps instances sequentially, batches arrays, resets finished environments, preserves terminal metadata |

- LunarLander is the **task environment**; Box2D is its internal **physics engine**.
- Evaluation uses separate environments with the same per-environment wrapper chain.
  Our command's `--n-eval-envs 1` selects one evaluation instance.



### Vectorized environments: batched interaction

$$\text{rollout transitions}=n_{\text{envs}}\times n_{\text{steps}}$$

```text
                  ┌─ LunarLander 0
PPO ↔ SB3 VecEnv ──┼─ LunarLander 1
                  ├─ ...
                  └─ LunarLander 15
```

| Interface | reset | step |
|---|---|---|
| Gymnasium Env | `obs, info` | `obs, reward, terminated, truncated, info` |
| SB3 VecEnv | `obs` | `obs, rewards, dones, infos` |

- SB3 automatically resets finished environments.
- `infos[i]["terminal_observation"]` preserves the observation before reset.
- `infos[i]["TimeLimit.truncated"]` distinguishes timeout handling.
- Batched does not necessarily mean multiprocessing: `DummyVecEnv` steps sequentially.


In [ ]:
vec_env = make_vec_env(ENV_ID, n_envs=2, seed=0)
try:
    observations = vec_env.reset()
    actions = np.array([0, 2])
    next_observations, rewards, dones, infos = vec_env.step(actions)
    print("Observations:", observations.shape)
    print("Actions:", actions.shape, "Rewards:", rewards.shape, "Dones:", dones.shape)
finally:
    vec_env.close()

## 4 · PPO in Stable-Baselines3

### Policy-gradient reminder

$$
J(\theta)=\mathbb{E}_{S_0\sim\mu}\!\left[V^{\pi_\theta}(S_0)\right]
=\mathbb{E}_{\mu,\pi_\theta}\!\left[\sum_{t=0}^{T-1}\gamma^t R_{t+1}\right].
$$

$$
\theta_{n+1}=\theta_n+\alpha_n\nabla_\theta J(\theta_n).
$$

- $J(\theta)$ is the value of the parameterized policy, averaged over initial states from $\mu$.
- $\alpha_n$ is the learning rate; in practice the gradient is estimated from sampled trajectories.
- PPO uses a clipped surrogate objective for the policy update.
- The critic $V_\phi$ estimates state values; it is distinct from the objective $J(\theta)$.

### PPO training loop

~~~text
initialize policy and value network
repeat:
    collect_rollouts(): interact with environments; compute advantages
    train(): update parameters using rollout minibatches for several epochs
~~~

For LunarLander, `MlpPolicy` maps eight state components to a distribution over
four actions and a state-value estimate.

**Hyperparameters:** open [hyperparams/ppo.yml](../../hyperparams/ppo.yml),
entry `LunarLander-v3`. Missing values use SB3 defaults; CLI overrides take precedence.


### The two methods to follow

**`OnPolicyAlgorithm.collect_rollouts()`**

- Evaluate the policy to obtain actions, value estimates and log probabilities.
- Call `env.step()` and store transitions in `RolloutBuffer`.
- Compute advantage estimates and return targets for the update.

Open [on_policy_algorithm.py](../../.venv/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py)
at `collect_rollouts()`. Its `learn()` method alternates collection and training.

**`PPO.train()`**

- Read minibatches from the rollout buffer for `n_epochs`.
- Compute the clipped policy loss, value loss and entropy term.
- Backpropagate the loss and update network parameters.

Open [ppo.py](../../.venv/lib/python3.12/site-packages/stable_baselines3/ppo/ppo.py)
at `train()`; locate `ratio`, `policy_loss` and `optimizer.step()`.


### The role of GAE λ

- GAE uses rewards and value estimates to estimate the advantage of sampled actions. 
- The advantage estimates replace the return-to-go estimates of the usual policy gradient.
- `gae_lambda` controls how far future TD residuals contribute.
- Find the LunarLander value in the Zoo configuration and compare it with the SB3 default.

Implementation: [buffers.py](../../.venv/lib/python3.12/site-packages/stable_baselines3/common/buffers.py),
`RolloutBuffer.compute_returns_and_advantage()`.

### PPO's clipped policy objective

$$
r_t(\theta)=\frac{\pi_\theta(A_t\mid S_t)}
{\pi_{\theta_{\mathrm{old}}}(A_t\mid S_t)}.
$$

$$
L^{\mathrm{CLIP}}(\theta)=\mathbb{E}_t\left[
\min\left(r_t(\theta)\widehat A_t,\,
\mathrm{clip}(r_t(\theta),1-\epsilon,1+\epsilon)\widehat A_t\right)
\right].
$$

- $r_t$ compares the current policy with the policy that collected the rollout.
- Clipping limits the incentive for large probability-ratio changes.
- SB3 minimizes the negative surrogate, together with value and entropy losses.

Open [PPO.train()](../../.venv/lib/python3.12/site-packages/stable_baselines3/ppo/ppo.py):
`ratio = exp(log_prob - old_log_prob)` corresponds to $r_t(\theta)$.


## 5 · Why RL Baselines3 Zoo?

- SB3 implements learning algorithms.
- A complete experiment also needs configuration, environment creation, seeds, evaluation, logging, and saved artifacts.
- Zoo connects these components through `ExperimentManager`.

```text
train.py
  └── rl_zoo3.train.train()
        ├── parse CLI arguments
        └── ExperimentManager(...)
              ├── setup_experiment()
              │     ├── read_hyperparameters()
              │     ├── preprocess hyperparameters
              │     ├── create log folder + callbacks
              │     ├── create_envs()
              │     └── ALGOS["ppo"](...) → SB3 PPO
              ├── learn() → model.learn()
              └── save_trained_model()
```

- Evaluation environments are created by the callback setup when evaluation is enabled.
- Hyperparameter preprocessing turns configuration strings into schedules, wrappers, and Python objects.


### Read the checked-out LunarLander configuration

| Parameter | Interpretation |
|---|---|
| `n_envs` | Number of environments |
| `n_steps` | Steps per environment per rollout |
| `batch_size` | Transitions per minibatch |
| `n_epochs` | Optimization epochs per rollout |
| `gamma` | Discount factor |
| `gae_lambda` | GAE parameter |
| `ent_coef` | Entropy-loss coefficient |

- YAML contains both Zoo settings and SB3 constructor arguments.
- Missing SB3 arguments use the installed constructor defaults.
- CLI `--hyperparams key:value` overrides the selected YAML configuration.


In [ ]:
config = yaml.safe_load((ROOT / "hyperparams" / "ppo.yml").read_text())[ENV_ID]
display(pd.DataFrame(config.items(), columns=["Parameter", "Zoo setting"]))
print("Transitions per rollout:", config["n_envs"] * config["n_steps"])

## 6 · Configure your own Zoo experiment

Open [train_ppo_example.sh](scripts/train_ppo_example.sh). It contains a short,
runnable PPO example on CartPole. Read `python train.py --help` and
[hyperparams/ppo.yml](../../hyperparams/ppo.yml) to understand the settings.

| Flag | Purpose |
|---|---|
| `--algo` | Learning algorithm |
| `--env` | Registered environment ID |
| `--n-timesteps` | Total training transitions across all environments |
| `--seed` | Random seed for this training run |
| `--eval-freq` | Evaluation interval in training transitions |
| `--eval-episodes` | Episodes per evaluation |
| `--n-eval-envs` | Separate evaluation environments |
| `--hyperparams` | Overrides of the environment's YAML settings |
| `-f` | Output root shared by this assignment's runs |

PPO collects complete rollouts, so the actual step count can exceed the requested
budget. Keep the output folder and evaluation settings unchanged for the final plots.

### Debug with the VS Code workspace

Open [RLII_26.code-workspace](../../RLII_26.code-workspace) in VS Code using
**File → Open Workspace from File**. In **Run and Debug**, select
**Lecture 01: PPO LunarLander (debug)**.

- The debugger starts the repository's `train.py` directly using `.venv`.
- This separate, short debugging example writes logs to `logs/lecture_01/debug_ppo`.
- Execution pauses on entry. Set breakpoints, then press F5 to continue.
- `justMyCode: false` enables stepping into SB3 and Gymnasium.

| Stop | Inspect |
|---|---|
| `ExperimentManager.setup_experiment()` | Configuration, environment and PPO construction |
| `OnPolicyAlgorithm.collect_rollouts()` | State batch, sampled actions, values |
| `PPO.train()` | Minibatch, probability ratio and policy loss |

The [debugging guide](DEBUGGING.md) lists five breakpoint locations and useful variables.

### Artifacts: more than policy weights

```text
logs/<experiment>/ppo/LunarLander-v3_<run-id>/
├── LunarLander-v3.zip           final model
├── best_model.zip              best mean evaluation return
├── evaluations.npz             evaluation times and per-episode returns
├── <env-index>.monitor.csv     training episode statistics
├── rl_model_<steps>_steps.zip   optional checkpoints
└── LunarLander-v3/
    ├── args.yml
    ├── command.txt
    └── config.yml
```

- `best_model.zip` and `evaluations.npz` require evaluation.
- The run ID is an experiment counter, **not** the random seed.
- A checkpoint records training state at a particular transition count.
- Zoo stores the command/configuration; also retain the Git commit, dirty changes, and dependency versions.
- Seeds improve repeatability but do not guarantee bitwise equality across devices and versions.


## Source navigation

| Open file | Locate |
|---|---|
| [train.py](../../train.py) | Entry point |
| [rl_zoo3/train.py](../../rl_zoo3/train.py) | CLI parsing and experiment orchestration |
| [exp_manager.py](../../rl_zoo3/exp_manager.py) | `setup_experiment()`, `create_envs()`, `create_callbacks()`, `learn()` |
| [ppo.yml](../../hyperparams/ppo.yml) | `LunarLander-v3` |
| [on_policy_algorithm.py](../../.venv/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py) | `collect_rollouts()` and `learn()` |
| [ppo.py](../../.venv/lib/python3.12/site-packages/stable_baselines3/ppo/ppo.py) | `PPO.train()` |

Library links target this course's Python 3.12 `.venv` on macOS/Linux.
On Windows, the corresponding files are under `.venv/Lib/site-packages/`.
Open them in the IDE; if Jupyter hides `.venv`, use the IDE's file navigation.
The [debugging guide](DEBUGGING.md) lists breakpoint locations for the installed versions.


## 7 · **TODO**: Assignment: PPO across training seeds

1. Adapt `scripts/train_ppo_example.sh` to train **PPO on LunarLander-v3** with
   **one million training timesteps per run**. Determine the required CLI changes yourself.
2. Choose **at least three distinct, nonnegative training seeds**. Adapt the seed
   setting and run the script once per seed. Keep all other settings identical.
   You may automate repetition, but no finished command or seed loop is provided here.
3. Keep `-f logs/lecture_01/ppo_assignment` and the evaluation settings unchanged.
   Run sequentially and wait for each training process to finish successfully.
4. Inspect each run's model, `args.yml`, `config.yml`, and `evaluations.npz`.
   Record the seeds, commands, Git revision, local changes and package versions.
5. Execute the final cells below. Compare learning speed, final return and
   variability. Why is a single seed insufficient?

With the project environment activated, launch your **adapted** file from the repository root:

```bash
bash course/lecture_01/scripts/train_ppo_example.sh
```

On Windows use a Bash-capable terminal (e.g. Git Bash or WSL) with the project
Python available. The script runs one experiment per invocation.

## 8 · PPO on LunarLander: mean evaluation return and standard error

The final cells create one plot directly from your completed assignment runs in
`logs/lecture_01/ppo_assignment`. CartPole smoke tests are excluded. The table
identifies the selected runs; for repeated seeds, the latest completed numbered
run is used.

First, evaluation episode returns are averaged within each training seed. The
single curve shows the mean of these seed means. The shaded region is ±1 standard
error of that mean: the sample standard deviation across seed means divided by
the square root of the number of seeds. It is not a 95% confidence interval.
Only evaluation timesteps shared by all selected seeds are included. No individual
seed curves or training plots are generated.


In [ ]:
import importlib.util
import matplotlib.pyplot as plt

spec = importlib.util.spec_from_file_location(
    "lecture_plots", COURSE / "scripts" / "visualize_ppo_lunarlander.py"
)
lecture_plots = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lecture_plots)
assignment_root = ROOT / "logs" / "lecture_01" / "ppo_assignment"
try:
    selected_runs = lecture_plots.assignment_runs(assignment_root)
except ValueError as exc:
    selected_runs = {}
    print(exc)
else:
    display(pd.DataFrame([
        {"seed": seed, "run": str(run.relative_to(ROOT))}
        for seed, run in selected_runs.items()
    ]))

In [ ]:
if selected_runs:
    figures = lecture_plots.plot_assignment(selected_runs)
    output = assignment_root / "plots"
    output.mkdir(parents=True, exist_ok=True)
    for name, fig in figures:
        fig.savefig(output / name, dpi=160, bbox_inches="tight")
        display(fig)
        plt.close(fig)
    print("Saved plots:", output)
else:
    print("Complete the assignment runs, then rerun the selection and plotting cells.")